# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Access top-level metadata properties
print(f"Dataset name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}\n")
print(f"Dataset ID: {dataset.metadata.id}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their fields
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets detected in schema metadata. Trying dataset.records() to auto-detect...")
    # Use dataset.records() with no argument to show auto-selected record sets
    try:
        sample_records = dataset.records()
        first = next(sample_records)
        print(f"Sample record: {first}")
        print("You may use dataset.records() directly if @id for record set is absent from metadata.")
    except Exception as e:
        print("Unable to read any records: ", e)
else:
    print("Available Record Sets:")
    for rs in record_sets:
        print(f"- RecordSet name: {getattr(rs, 'name', 'N/A')}, id: {rs.id}")
        if hasattr(rs, 'fields'):
            print("  Fields:")
            for f in rs.fields:
                print(f"    - {f.get('name', f.get('@id', 'N/A'))} (id: {f.get('@id', 'N/A')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Attempt to extract all record sets. If not available, fallback to records() generator.

import warnings
warnings.filterwarnings("ignore")

# Use metadata to get record set @id. If empty, fallback:
record_set_ids = []
if dataset.metadata.record_sets:
    for rs in dataset.metadata.record_sets:
        record_set_ids.append(rs.id)
else:
    # Fallback: Use auto-detection by the mlcroissant backend
    # (mlcroissant will usually select the top-level data table.)
    print("No explicit record_set ids found; will try dataset.records() with no argument.")
    record_set_ids = [None]  # None will invoke default

dataframes = {}
for rs_id in record_set_ids:
    print(f"Extracting records from record_set: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id)) if rs_id is not None else list(dataset.records())
        dataframes[rs_id if rs_id is not None else 'default'] = pd.DataFrame(records)
        display_keys = dataframes[rs_id if rs_id is not None else 'default'].columns.tolist()
        print(f"Fields/columns in DataFrame: {display_keys}")
        display(dataframes[rs_id if rs_id is not None else 'default'].head())
    except Exception as e:
        print(f"Unable to load records for record_set: {rs_id or 'default'}: {e}")
    print("---")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, select a numeric column and a group field by previewing columns.
# Please adjust field names as appropriate based on the output above (e.g., 'MW_kDa', 'PeptideCount', 'Abundance_SampleA').

# Use the default record set if there is only one
if 'default' in dataframes:
    df = dataframes['default']
else:
    # pick the first one
    key0 = list(dataframes.keys())[0]
    df = dataframes[key0]

print("Available columns:", df.columns.tolist())

# Choose a numeric field. Adjust as needed (e.g. 'MW_kDa', 'Coverage_Percent', 'PeptideCount')
numeric_field = None
for candidate in ['MW_kDa', 'Coverage_Percent', 'PeptideCount', 'Abundance_SampleA']:
    if candidate in df.columns:
        numeric_field = candidate
        break
if not numeric_field:
    numeric_field = df.select_dtypes(include='number').columns[0] if not df.select_dtypes(include='number').empty else df.columns[0]
    print(f"Could not auto-detect standard numeric field, using: {numeric_field}")

threshold = df[numeric_field].mean() # e.g., filter to above-average if not sure
filtered_df = df[df[numeric_field].astype(float) > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()
) / filtered_df[numeric_field].astype(float).std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a main categorical field (e.g., 'Description', 'SampleType', etc)
group_field = None
for candidate in ['Description', 'Protein', 'SampleType', 'Group']:
    if candidate in df.columns:
        group_field = candidate
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field} (mean of numeric fields):")
    display(grouped_df.head())
else:
    print("No suitable group categorical field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].astype(float), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If there's another numeric field, scatter plot
numeric_candidates = [col for col in df.select_dtypes(include='number').columns if col != numeric_field]
if numeric_candidates:
    plt.figure(figsize=(7,5))
    other_field = numeric_candidates[0]
    sns.scatterplot(x=df[numeric_field].astype(float), y=df[other_field].astype(float))
    plt.xlabel(numeric_field)
    plt.ylabel(other_field)
    plt.title(f'Scatter plot of {numeric_field} vs {other_field}')
    plt.show()
else:
    print("Not enough numeric fields for scatter plot.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the FAIR^2 dataset for Mass Spectrometry Analysis of Extracellular Vesicles using the Croissant schema and the `mlcroissant` library.
- We explored its structure and loaded the main data table as a pandas DataFrame.
- Basic exploratory analysis included filtering by a numeric field, normalization, and optional grouping.
- Distributions and variable relationships were visualized. For deeper analysis, consult the field and column names using the DataFrame's `.columns` and refer to the croissant schema documentation.

For additional analysis, consider examining specific protein IDs, comparing samples, or integrating with downstream bioinformatics workflows.